# Orchestration & Automation with Databricks Workflows

Multi-task jobs, dependencies, scheduling, retries, and repair runs — the essentials of pipeline orchestration.

| Training Block | Duration | Type |
|---|---|---|
| Orchestration & Automation — Demo | 25 min | Demo |

**Prerequisites:** 03 — Lakeflow Pipelines (understanding of pipeline concepts)

## Learning Objectives

After completing this module you will be able to:

- **Create** multi-task Databricks Workflows with task dependencies
- **Configure** scheduling patterns (CRON expressions)
- **Apply** retry policies and understand repair runs
- **Use** `dbutils.widgets` and `dbutils.jobs.taskValues` for parameterization
- **Monitor** job execution via UI and system tables

## Introduction to Lakeflow Jobs

**Lakeflow Jobs** (formerly Databricks Jobs) is a fully managed orchestration service that lets you define **multi-task workflows** as directed acyclic graphs (DAGs). Each task can be a notebook, pipeline, SQL query, Python script, or even a conditional branch.

A Job run follows this lifecycle:

```
TRIGGER → QUEUED → RUNNING → [RETRY if transient error] → SUCCESS / FAILED
                                                            ↓
                                                      REPAIR RUN (re-run failed + downstream only)
```

| Scenario | Solution |
|----------|----------|
| ETL pipeline with multiple steps | Multi-task Job |
| Daily report at fixed time | Scheduled Job (CRON) |
| Reaction to new files | File Arrival Trigger |
| Reaction to upstream table update | Table Update Trigger |
| ML training pipeline | Job with notebook tasks |
| Run Lakeflow Pipeline | Job with Pipeline task |
| Conditional branching | If/Else task |
| Parameterized batch runs | For Each task |

---

| Feature | Jobs | Lakeflow Pipelines |

|---------|------|-----|**Best Practice**: Use Lakeflow Pipelines for ETL transformations, Jobs for orchestrating Pipelines + other tasks (ML, reports, notifications).

| Orchestration | General-purpose (any task type) | ETL-focused (SQL/Python) |

| Dependencies | Manual configuration (DAG) | Automatic (lineage-based) || Flexibility | High | Opinionated |

| Data Quality | Custom code | Built-in expectations || Compute | Per-task cluster or Serverless | Shared pipeline cluster |
| Error Handling | Retry + Repair Runs | Auto-retry within pipeline |

### Task Types in Lakeflow Jobs

Overview of all available task types in Lakeflow Jobs and when to use each one.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| **Notebook** | Run a Databricks notebook | ETL logic, ML training |
| **Pipeline** | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| **Python Script** | Run a `.py` file from Workspace/Repos | Utility scripts, data validation |
| **SQL** | Run a SQL query or file | DDL, reporting queries, CTAS |
| **JAR** | Run a Java/Scala JAR | Legacy Spark jobs |
| **Spark Submit** | Submit a Spark application | Custom Spark apps, external JARs |
| **If/Else Condition** | Branch based on task value or parameter | Conditional workflows |
| **For Each** | Iterate over a JSON array | Parameterized batch runs (e.g., per-region) |

#### Repair Runs

When a Job fails mid-execution, you don't have to re-run the entire pipeline. **Repair Runs** re-execute only the **failed tasks and their downstream dependents**, skipping all previously successful tasks.

> **Pro Tip:** If/Else conditions evaluate `{{tasks.<task_key>.result_state}}` or task values. For Each iterates over a JSON array and runs nested tasks for each element — great for multi-region or multi-table patterns.

```

Run #1:  validate ✅ → transform ❌ → report (skipped)```

Repair:  validate (skipped ✅) → transform 🔄 → report 🔄databricks jobs repair-run <run-id> --rerun-tasks transform,report

```# Databricks CLI

```bash

This saves significant compute time and cost in large pipelines. You can trigger a Repair Run from the UI (Runs tab → Repair) or via API:

## Preparing Notebooks for Job

Below are 3 simple notebooks that we'll use in the demo.

**Instructions**: 
1. Create folder `/Workspace/Users/<your-email>/jobs_demo/`
2. Copy each of the following code snippets to a separate notebook

---

### Task 1: Validate Source

Validates row count against `min_rows` threshold. Returns status + count via `dbutils.notebook.exit()`.

In [0]:
# TASK 1: Validate Source Data
# Copy this code to notebook: task_01_validate

# Parameters from Job
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("min_rows", "100")

source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

# Validation
df = spark.table(source_table)
row_count = df.count()

print(f"Source: {source_table}")
print(f"Row count: {row_count}")
print(f"Minimum required: {min_rows}")

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

print("Validation PASSED")

# Return result to next task
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "source_table": source_table,
    "row_count": row_count
}))

### Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile). Returns row count.

**Task Values — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Set value (upstream task) | `dbutils.jobs.taskValues.set(key="row_count", value=1000)` |
| Get value (downstream task) | `val = dbutils.jobs.taskValues.get(taskKey="Task_1", key="row_count")` |
| Supported types | `str`, `int`, `float`, `bool`, JSON-serializable `dict`/`list` |
| Max value size | 48 KB per key |


In [0]:
# TASK 2: Transform Data
# Copy this code to notebook: task_02_transform

from pyspark.sql.functions import *
from datetime import date
import json

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("run_date", "")

source_table = dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(date.today())

# Get result from previous task (optional)
try:
    prev_result = dbutils.jobs.taskValues.get(
        taskKey="validate",
        key="returnValue",
        default="{}"
    )
    prev_data = json.loads(prev_result)
    print(f"Previous task result: {prev_data}")
except:
    print("Running standalone (no previous task)")

# Transformation
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("trip_duration_minutes", 
                round((col("tpep_dropoff_datetime").cast("long") - 
                       col("tpep_pickup_datetime").cast("long")) / 60, 2))
    .withColumn("cost_per_mile", 
                when(col("trip_distance") > 0, 
                     round(col("fare_amount") / col("trip_distance"), 2))
                .otherwise(0))
    .withColumn("processing_date", lit(run_date))
)

row_count = df_transformed.count()
print(f"Transformed {row_count} rows")

df_transformed.select(
    "trip_distance", "fare_amount", "trip_duration_minutes", "cost_per_mile"
).show(5)

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "rows_transformed": row_count
}))

### Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance). Prints summary report.

In [0]:
# TASK 3: Generate Report
# Copy this code to notebook: task_03_report

from pyspark.sql.functions import *
import json
from datetime import datetime

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")

source_table = dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_trips"),
    round(sum("fare_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(max("fare_amount"), 2).alias("max_fare")
).collect()[0]

# Display report
print("\n" + "="*50)
print("DAILY REPORT")
print("="*50)
print(f"Total Trips:    {report.total_trips:,}")
print(f"Total Revenue:  ${report.total_revenue:,.2f}")
print(f"Avg Fare:       ${report.avg_fare:.2f}")
print(f"Avg Distance:   {report.avg_distance:.2f} miles")
print(f"Max Fare:       ${report.max_fare:.2f}")
print("="*50)
print(f"Generated at:   {datetime.now()}")
print("="*50 + "\n")

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "total_trips": report.total_trips,
    "total_revenue": float(report.total_revenue)
}))

### [UI DEMO] Creating Multi-task Job

**Step 1: Create new Job**
- [ ] Workflows → Create Job
- [ ] Name: `Demo_ETL_Pipeline`

<img src="../../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">


- [ ] Cluster Job: Create new cluster job

<img src="../../../assets/images/a967557a143a40c0ac7ed26ce469866a.png" width="800">

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `/Workspace/.../task_01_validate`
- [ ] Cluster: Serverless or new Job Cluster
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

<img src="../../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `/Workspace/.../task_02_transform`
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `/Workspace/.../task_03_report`

<img src="../../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Run now
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output

---

## [UI DEMO 2] Medallion Pipeline Job — Ready-to-Use Config

We already have 6 production-ready notebooks in `materials/medallion/` that implement a full Medallion pipeline:

| Layer | Notebook | Input | Output |
|-------|----------|-------|--------|
| **Bronze** | `bronze_customers` | CSV files | `bronze.bronze_customers` |
| **Bronze** | `bronze_orders` | JSON files | `bronze.bronze_orders` |
| **Silver** | `silver_customers` | bronze_customers | `silver.silver_customers` |
| **Silver** | `silver_orders_cleaned` | bronze_orders | `silver.silver_orders_cleaned` |
| **Gold** | `gold_customer_orders_summary` | silver_customers + silver_orders | `gold.gold_customer_orders_summary` |
| **Gold** | `gold_daily_orders` | silver_orders | `gold.gold_daily_orders` |

**DAG Structure:**
```
bronze_customers ──→ silver_customers ──────→ gold_customer_orders_summary
bronze_orders ────→ silver_orders_cleaned ─┤
 └→ gold_daily_orders
```

**How to deploy:** Copy the JSON config below → Workflows → Create Job → switch to JSON editor (or use Databricks CLI: `databricks jobs create --json @job.json`).

---

### Declarative Automation Bundles (DABs) — YAML Configuration

Below is the **Declarative Automation Bundles** (formerly Databricks Asset Bundles / DABs) YAML definition for the Medallion pipeline Job. This was exported from a working Databricks deployment.

> **Before deploying:** Update `notebook_path`, `existing_cluster_id`, and `parameters` to match your environment.

```yaml
resources:
  jobs:
    job_demo_2:
      name: job_demo_medalion
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_orders_cleaned
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_daily_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: silver_customer
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE
          environment_key: Default
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
      environments:
        - environment_key: Default
          spec:
            environment_version: "5"
      performance_target: PERFORMANCE_OPTIMIZED

```

| Element | Description |
|---------|-------------|
| `existing_cluster_id` | Use existing all-purpose cluster (for demo) or replace with `job_cluster_key` for production |
| `depends_on` | Defines task dependencies — creates the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

### Deploy Checklist

**Option A — JSON Editor (UI):**
1. [ ] Workflows → Create Job
2. [ ] Click ` Edit as JSON` (top-right)
3. [ ] Paste the JSON above (replace placeholders)
4. [ ] Save → Run now

**Option B — Databricks CLI:**
```bash
# Save JSON to file, then:
databricks jobs create --json @medallion_pipeline_job.json
```

**After deployment — show participants:**
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Task-level parameters (catalog, schema, source_path)
- [ ] Trigger: set to PAUSED — run manually for demo
- [ ] Run → observe sequential layer execution (Bronze → Silver → Gold)
- [ ] Show Repair Run: intentionally fail one task → repair reruns only failed + downstream

---

## [UI DEMO] Triggers and Schedule

Lakeflow Jobs supports multiple trigger types. A Job can have **multiple triggers** simultaneously (e.g., scheduled + file arrival). Each trigger creates an independent run.

---

| Trigger Type | Behavior | Example | Max Concurrency |
|---|---|---|---|
| **Scheduled (CRON)** | Fixed schedule using Quartz CRON | `0 0 2 * * ?` — daily at 2:00 | Configurable |
| **File arrival** | Fires when new files land in a UC Volume or cloud path | New CSV in `/Volumes/.../landing/` | 1 (default) |
| **Table update** | Fires when an upstream Delta table is updated | `catalog.schema.bronze_orders` modified | 1 (default) |
| **Continuous** | Restarts immediately after completion | Streaming-like always-on | 1 |
| **Manual** | On-demand via UI, CLI, or API | Testing, ad-hoc runs | Configurable |


#### CRON Syntax (Quartz format)> **Pro Tip:** `?` means "no specific value" — use it in either day-of-month OR day-of-week (not both). File arrival triggers check for new files every ~60 seconds.



Databricks uses **Quartz CRON** with 6-7 fields: `second minute hour day-of-month month day-of-week [year]`| Day of week | 1-7 or SUN-SAT | `, - * ? / L #` |

| Month | 1-12 or JAN-DEC | `, - * /` |

| Field | Allowed Values | Special Characters || Day of month | 1-31 | `, - * ? / L W` |

|-------|---------------|--------------------|| Hour | 0-23 | `, - * /` |

| Second | 0-59 | `, - * /` || Minute | 0-59 | `, - * /` |


### Trigger Configuration Checklist

Step-by-step instructor checklist for demonstrating trigger options in the Lakeflow Jobs UI.

**Trigger Options** (Triggers tab):

| Trigger Type | Usage | Example |
|--------------|-------|---------|
| **Scheduled** | Fixed schedule | Daily at 2:00 |
| **File arrival** | Reaction to new files | New file in `/landing/` |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

<img src="../../../assets/images/6e23746ce063491fa8afb3dea6268a1d.png" width="800">


**Demo: Scheduled Trigger**
- [ ] Add trigger → Scheduled
- [ ] Cron expression: `0 0 2 * * ?` (daily at 2:00)
- [ ] Timezone: `Europe/Warsaw`
- [ ] Show: Preview next runs


<img src="../../../assets/images/cf3cfc85162a466eb77e15e20df5c15c.png" width="800">

**Demo: File Arrival Trigger** (optional)
- [ ] Add trigger → File arrival
- [ ] URL: Unity Catalog Volume path
- [ ] Min files: 1
### Useful CRON Expressions

Common CRON patterns for scheduling Lakeflow Jobs at various intervals.

```
0 0 2 * * ? # Daily at 2:00
0 0 * * * ? # Every hour
0 0 9 ? * MON-FRI # Mon-Fri at 9:00
0 0 0 1 * ? # First day of month
0 */15 * * * ? # Every 15 minutes
```

---

## Serverless Compute for Jobs

Since 2025, Databricks offers **Serverless compute** as the default option for Lakeflow Jobs. Serverless eliminates cluster management — tasks start in seconds with auto-scaled, ephemeral compute.

| Aspect | Classic Clusters | Serverless Compute |
|--------|-----------------|-------------------|
| **Startup time** | 3-10 minutes | Seconds |
| **Cluster management** | Manual sizing (workers, instance types) | Fully automatic |
| **Cost model** | Per-VM (DBU + cloud infra) | Per-DBU only (no infra cost) |
| **Spot instances** | Configurable (cost savings) | Not applicable |
| **Auto-scaling** | Configurable (min/max workers) | Built-in, transparent |
| **Custom libraries** | Full control (init scripts, pip) | Limited (pip only, no init scripts) |
| **GPU support** | Yes | No |
| **Use case** | Long-running, GPU, custom libs, cost-sensitive batch | ETL, SQL, notebooks, triggered jobs |

**How to enable:** In the Job task configuration, select **Serverless** as the compute type instead of specifying a cluster.


#### When to Use Classic Clusters Instead---



| Scenario | Why Classic |> **Pro Tip:** Serverless compute is ideal for **triggered jobs** that need fast response times (file arrival, table update). For cost-sensitive scheduled batch jobs running for hours, classic clusters with spot instances may still be 30-50% cheaper.

|----------|-------------|

| GPU / ML training | Serverless doesn't support GPU instances || Network isolation (Private Link) | Classic clusters support custom VNet/VPC |

| Custom init scripts | Serverless only supports pip packages || Specific instance types needed | Serverless auto-selects instance types |
| Long-running batch (hours) | Spot instances on classic can be significantly cheaper |

## Summary

| Topic | Key Concept | Key Terms |
|---|---|---|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Continuous | `0 0 2 * * ?` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |

---

← [03 — Lakeflow Pipelines](03_lakeflow_pipelines_demo.ipynb) | **[ README](../../../README.md)**
